# Full run: results on the real models

Every SWE-bench Lite item and 300 LoCoMo questions, through each condition, on **Llama 3.1 8B** (vLLM on a Modal H100) and **Muse Spark 1.2** (Meta Model API). Then the team replay.

**The north star:** for a team sharing one codebase, the gateway matches the accuracy of sending context straight to Muse while sending **≤ 10%** of Muse's tokens. The pass rule is **within 3 points of `muse_direct`, at ≤ 10% of its Muse tokens**.

| Condition | What Muse sees |
|---|---|
| `muse_direct` | the whole context, no gateway (SWE-bench: ~11K tokens of retrieved code; LoCoMo: the full conversation, ~18K tokens) |
| `full` | the learned gate (after `scripts/gate_run.sh`): Llama drafts, the gate keeps the draft or escalates |
| `escalate_raw` | always escalates, with the raw pages behind the top-5 memories (after `scripts/gate_run.sh`) |
| `escalate_compact` | always escalates, with the top-5 memory summaries |
| `rule_gate` | the hand-written gate: escalates long or code-heavy questions |
| `llama_only` | never: Llama answers with memory |

Sections:
1. What was run
2. The north star, per benchmark
3. Cost and latency
4. LoCoMo: where the gap comes from
5. SWE-bench: what its accuracy measures
6. Routing headroom: the best any gate could do
7. Team scaling (replay)
8. The learned gate
9. Summary

Everything is read from `eval/results/`. Nothing here calls a model or needs the database. `full` and `escalate_raw` appear in every table once `scripts/gate_run.sh` has run; just rerun the notebook.

In [ ]:
RUN_ID = "full1"   # the FULL_ID scripts/full_run.sh used
N, SEED = 300, 0   # its FULL_N, and eval.run's default seed

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "eval").is_dir() else Path.cwd().parent
os.chdir(ROOT)  # so .env and the repo's packages are found
sys.path.insert(0, str(ROOT))

from eval import locomo  # noqa: E402
from gateway.config import Settings  # noqa: E402

RESULTS = ROOT / "eval" / "results"
BENCHES = ["swebench", "locomo"]
ORDER = ["muse_direct", "full", "escalate_raw", "escalate_compact", "rule_gate", "llama_only"]
HEADLINE = {"swebench": "hit", "locomo": "correct"}
CATEGORIES = {1: "multi-hop", 2: "temporal", 3: "open-domain", 4: "single-hop"}
MARGIN, BUDGET = 3.0, 0.10  # points of accuracy, share of muse_direct's Muse tokens
settings = Settings()
PRICE_IN = settings.muse_input_price_per_m / 1e6
PRICE_OUT = settings.muse_output_price_per_m / 1e6

# Validated colours (dataviz reference palette; all three pass all-pairs CVD checks).
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
SURFACE, INK, INK_2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK_2, "xtick.color": MUTED, "ytick.color": MUTED,
    "text.color": INK, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.titlesize": 12,
    "axes.titleweight": "bold", "font.family": ["Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 110,
})
pd.set_option("display.max_colwidth", 80)


def run_id(bench: str, condition: str) -> str:
    return f"{RUN_ID}-{bench}-{condition}"


runs = {}  # (bench, condition) -> summary
for bench in BENCHES:
    for c in ORDER:
        path = RESULTS / f"{bench}_{c}_{run_id(bench, c)}.json"
        if path.exists():
            runs[bench, c] = json.loads(path.read_text())
conditions = {b: [c for c in ORDER if (b, c) in runs] for b in BENCHES}

rows = []
for (bench, c) in runs:
    path = RESULTS / "raw" / f"{bench}_{c}_{run_id(bench, c)}.jsonl"
    rows += [{**json.loads(line), "bench": bench} for line in path.read_text().splitlines()]
df = pd.DataFrame(rows)
df["correct"] = [bool(s.get(HEADLINE[b])) for s, b in zip(df.score, df.bench)]
df["muse_tokens"] = df.muse_prompt_tokens + df.muse_completion_tokens
df["muse_usd"] = df.muse_prompt_tokens * PRICE_IN + df.muse_completion_tokens * PRICE_OUT
df["escalated"] = df.route.str.startswith("escalate") | (df.route == "muse_direct")
category = {item.id: CATEGORIES.get(item.meta["category"]) for item in locomo.load(N, SEED)}
df["category"] = df.item_id.map(category)


def wide(bench: str, column: str = "correct") -> pd.DataFrame:
    # One row per item answered by every condition, one column per condition.
    sub = df[df.bench == bench].pivot(index="item_id", columns="baseline", values=column)
    return sub[conditions[bench]].dropna()

## 1. What was run

One row per condition. Every result must come from the real models (not simulated, not a stand-in), at answer temperature 0, with the concurrent ingest validated by the 100-item check.

In [ ]:
KEYS = ["items", "llama_model", "muse_model", "simulated", "muse_is_standin", "answer_temperature",
        "ingest_concurrency", "offline_replay"]
provenance = pd.DataFrame([{"bench": b, "condition": c, **{k: runs[b, c].get(k) for k in KEYS}}
                           for b in BENCHES for c in conditions[b]])
real = not provenance.simulated.any() and not provenance.muse_is_standin.any()
common = {b: len(wide(b)) for b in BENCHES}
display(Markdown(
    f"**{'REAL models' if real else 'NOT the real models: these are not results'}.** "
    + ", ".join(f"{b}: {common[b]} items answered by all {len(conditions[b])} conditions" for b in BENCHES)
    + ". `offline_replay` marks the learned gate, scored with each route's recorded answers."))
provenance

## 2. The north star, per benchmark

- **gap**: accuracy minus `muse_direct`'s, in points. The 95% interval comes from a paired bootstrap over items, so every resample uses the same items for every condition.
- **token share**: Muse tokens (prompt + completion) as a share of `muse_direct`'s on the same items. **input share** counts prompt tokens only. Muse is a reasoning model, so every escalation costs ~1,000–3,000 output tokens however short its prompt. The input share shows how much context the gateway saved.
- **verdict**: the pass rule on the point estimates. It's flagged when the interval still crosses −3. **passes on input** applies the same rule with the input share.

In [ ]:
rng = np.random.default_rng(0)
B = 4000
rows = []
for bench in BENCHES:
    correct = wide(bench).astype(float)
    idx = rng.integers(0, len(correct), size=(B, len(correct)))
    boot = {c: correct[c].to_numpy()[idx].mean(axis=1) * 100 for c in correct}
    sub = df[(df.bench == bench) & df.item_id.isin(correct.index)]
    tokens = sub.groupby("baseline")[["muse_prompt_tokens", "muse_tokens"]].sum()
    for c in conditions[bench]:
        gap = boot[c] - boot["muse_direct"]
        rows.append({
            "bench": bench, "condition": c, "accuracy": correct[c].mean() * 100,
            "acc_lo": np.percentile(boot[c], 2.5), "acc_hi": np.percentile(boot[c], 97.5),
            "gap": (correct[c].mean() - correct["muse_direct"].mean()) * 100,
            "gap_lo": np.percentile(gap, 2.5), "gap_hi": np.percentile(gap, 97.5),
            "token_share": tokens.muse_tokens[c] / tokens.muse_tokens["muse_direct"],
            "input_share": tokens.muse_prompt_tokens[c] / tokens.muse_prompt_tokens["muse_direct"]})
star = pd.DataFrame(rows).set_index(["bench", "condition"])


def verdict(r, share="token_share"):
    if r.name[1] == "muse_direct":
        return "reference"
    failed = [why for why, bad in (("accuracy", r.gap < -MARGIN), ("tokens", r[share] > BUDGET)) if bad]
    if failed:
        return "FAIL: " + " and ".join(failed)
    return "PASS" if r.gap_lo >= -MARGIN else "PASS (interval crosses −3)"


star["verdict"] = star.apply(verdict, axis=1)
star["passes on input"] = star.apply(lambda r: verdict(r, "input_share").startswith("PASS")
                                     if r.name[1] != "muse_direct" else None, axis=1)
star.style.format({"accuracy": "{:.1f}", "acc_lo": "{:.1f}", "acc_hi": "{:.1f}", "gap": "{:+.1f}",
                   "gap_lo": "{:+.1f}", "gap_hi": "{:+.1f}", "token_share": "{:.1%}",
                   "input_share": "{:.1%}"}, na_rep="")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7.2), gridspec_kw={"width_ratios": [1.15, 1]})
for (a1, a2), bench in zip(axes, BENCHES):
    t = star.loc[bench].drop("muse_direct")
    ys = np.arange(len(t))
    a1.axvspan(min(-MARGIN, t.gap_lo.min()) - 5, -MARGIN, color=GRID, alpha=0.6, lw=0)
    a1.axvline(0, color=AXIS, lw=1)
    a1.axvline(-MARGIN, color=MUTED, lw=1, ls="--")
    a1.errorbar(t.gap, ys, xerr=[t.gap - t.gap_lo, t.gap_hi - t.gap], fmt="o", color=BLUE,
                ecolor=BLUE, elinewidth=2, capsize=0, ms=8)
    for y, (c, r) in zip(ys, t.iterrows()):
        a1.annotate(f"{r.gap:+.1f}", (r.gap, y), xytext=(0, 8), textcoords="offset points",
                    ha="center", fontsize=9, color=INK_2)
    a1.set_yticks(ys, t.index)
    a1.invert_yaxis()
    a1.set_ylim(len(t) - 0.4, -0.7)
    a1.grid(axis="y", visible=False)
    ref = star.loc[(bench, "muse_direct"), "accuracy"]
    a1.set_title(f"{bench}: accuracy vs muse_direct ({ref:.1f}%), points", loc="left", fontsize=11)

    h = 0.36
    a2.barh(ys - h / 2, t.token_share * 100, height=h, color=BLUE, label="all Muse tokens")
    a2.barh(ys + h / 2, t.input_share * 100, height=h, color=ORANGE, label="prompt tokens only")
    for y, (c, r) in zip(ys, t.iterrows()):
        a2.text(r.token_share * 100, y - h / 2, f" {r.token_share:.1%}", va="center", fontsize=8, color=INK_2)
        a2.text(r.input_share * 100, y + h / 2, f" {r.input_share:.1%}", va="center", fontsize=8, color=INK_2)
    a2.axvline(BUDGET * 100, color=MUTED, lw=1, ls="--")
    a2.set_yticks(ys, [""] * len(t))
    a2.invert_yaxis()
    a2.set_xlim(0, max(30, t.token_share.max() * 120))
    a2.grid(axis="y", visible=False)
    a2.set_title("Muse tokens, % of muse_direct", loc="left", fontsize=11)
axes[0, 1].legend(frameon=False, fontsize=8, loc="lower right")
for ax in axes[1]:
    ax.set_xlabel("dashed line: the north star's limit")
plt.tight_layout()
plt.show()

## 3. Cost and latency

Per question. Dollars use the Muse prices in `.env`. Seconds are wall time per question as the harness saw it, including the round trip to Modal or the Meta API. **cached share** is the share of Llama's prompt tokens served from the prefix cache.

In [ ]:
g = df.groupby(["bench", "baseline"])
cost = pd.DataFrame({
    "accuracy": g.correct.mean() * 100,
    "escalated": g.escalated.mean() * 100,
    "muse_prompt": g.muse_prompt_tokens.mean(),
    "muse_completion": g.muse_completion_tokens.mean(),
    "usd_per_1k_questions": g.muse_usd.mean() * 1000,
    "median_s": g.seconds.median(),
    "p90_s": g.seconds.quantile(0.9),
    "llama_compute": g.compute_tokens.mean(),
    "cached_share": g.cached_tokens.sum() / g.llama_prompt_tokens.sum().replace(0, np.nan),
}).reindex([(b, c) for b in BENCHES for c in conditions[b]])
cost.style.format({"accuracy": "{:.1f}", "escalated": "{:.0f}%", "muse_prompt": "{:,.0f}",
                   "muse_completion": "{:,.0f}", "usd_per_1k_questions": "${:,.2f}",
                   "median_s": "{:.1f}", "p90_s": "{:.1f}", "llama_compute": "{:,.0f}",
                   "cached_share": "{:.0%}"}, na_rep="")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 0.45 * max(map(len, conditions.values())) + 1.6))
for ax, bench in zip(axes, BENCHES):
    t = cost.loc[bench]
    ys = np.arange(len(t))
    ax.barh(ys, t.muse_prompt, color=BLUE, height=0.55, label="prompt")
    ax.barh(ys, t.muse_completion, left=t.muse_prompt, color=ORANGE, height=0.55,
            edgecolor=SURFACE, linewidth=2, label="completion (incl. reasoning)")
    for y, (c, r) in zip(ys, t.iterrows()):
        ax.text(r.muse_prompt + r.muse_completion, y, f" {r.muse_prompt + r.muse_completion:,.0f}",
                va="center", fontsize=8, color=INK_2)
    ax.set_yticks(ys, t.index)
    ax.invert_yaxis()
    ax.grid(axis="y", visible=False)
    ax.set_xlim(0, (t.muse_prompt + t.muse_completion).max() * 1.2)
    ax.set_title(f"{bench}: Muse tokens per question", loc="left", fontsize=11)
axes[0].legend(frameon=False, fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

## 4. LoCoMo: where the gap comes from

LoCoMo is the benchmark where the gateway is furthest from `muse_direct`. Two things can lose an item:
- **Retrieval:** none of the top-5 memories comes from a session that holds the evidence (`source_in_hits`).
- **Compression or model:** the right session was retrieved, but its summary dropped the detail, or the model got it wrong.

`muse_direct` reads the whole conversation, so it is affected by neither.

In [ ]:
lo = df[(df.bench == "locomo") & df.item_id.isin(wide("locomo").index)]
by_category = lo.pivot_table(index="category", columns="baseline", values="correct",
                             aggfunc="mean").reindex(CATEGORIES.values())[conditions["locomo"]] * 100
by_category.insert(0, "items", lo[lo.baseline == "muse_direct"].groupby("category").size())
by_category.loc["all"] = [len(wide("locomo")), *(lo.groupby("baseline").correct.mean()[conditions["locomo"]] * 100)]
display(Markdown("**Accuracy (%) by question type**"))
display(by_category.style.format("{:.0f}").background_gradient(
    cmap="Blues", vmin=0, vmax=100, subset=conditions["locomo"]))

In [ ]:
retrieved = lo[lo.baseline == "llama_only"].set_index("item_id").source_in_hits.astype(bool)
memory_conditions = [c for c in conditions["locomo"] if c != "muse_direct"]
correct = wide("locomo").astype(bool)
split = pd.DataFrame({
    c: {"source retrieved: right": (correct[c] & retrieved).sum(),
        "source retrieved: wrong": (~correct[c] & retrieved).sum(),
        "not retrieved: right": (correct[c] & ~retrieved).sum(),
        "not retrieved: wrong": (~correct[c] & ~retrieved).sum()}
    for c in conditions["locomo"]}).T
segments = [("source retrieved: right", BLUE, None), ("source retrieved: wrong", ORANGE, None),
            ("not retrieved: right", BLUE, "////"), ("not retrieved: wrong", ORANGE, "////")]
fig, ax = plt.subplots(figsize=(10, 0.5 * len(split) + 1.4))
left = np.zeros(len(split))
for name, colour, hatch in segments:
    ax.barh(split.index, split[name], left=left, color=colour, hatch=hatch, height=0.55,
            edgecolor=SURFACE, linewidth=2, label=name)
    for y, (v, x) in enumerate(zip(split[name], left)):
        if v >= 12:
            ax.text(x + v / 2, y, str(v), ha="center", va="center", fontsize=8, color="white")
    left += split[name].to_numpy()
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
ax.set_xlabel("questions")
ax.legend(frameon=False, fontsize=8, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.22))
ax.set_title("LoCoMo: right or wrong, by whether memory from the evidence's session was retrieved",
             loc="left", fontsize=11)
plt.tight_layout()
plt.show()

rates = pd.DataFrame({c: {"accuracy, source retrieved": correct[c][retrieved].mean() * 100,
                          "accuracy, not retrieved": correct[c][~retrieved].mean() * 100}
                      for c in conditions["locomo"]}).T
display(Markdown(
    f"Retrieval finds the evidence's session for **{retrieved.mean():.0%}** of questions. "
    "The same item split applies to `muse_direct`, which shows how hard each subset is "
    "when nothing is lost."))
rates.round(1)

## 5. SWE-bench: what its accuracy measures

The task is to name the file to change. Its "evidence" is the gold file:
- **`muse_direct`:** the gold file is in view if it is among the ~11K tokens of code retrieved for the full context (`in_context`, the ceiling).
- **Gateway conditions:** it is in view if one of the top-5 memories comes from it (`gold_in_hits`).

When the gold file isn't in view, a right answer comes from the issue text (which often names the file or module) or from the model's knowledge of the repo.

In [ ]:
sw = df[(df.bench == "swebench") & df.item_id.isin(wide("swebench").index)].copy()
sw["evidence shown"] = np.where(sw.baseline == "muse_direct", sw.in_context,
                                sw.gold_in_hits.fillna(False)).astype(bool)
shown = sw.groupby(["baseline", "evidence shown"]).correct.agg(["mean", "size"]).unstack()
shown.columns = [f"{'accuracy' if m == 'mean' else 'items'}, {'shown' if s else 'not shown'}"
                 for m, s in shown.columns]
shown = shown.reindex(conditions["swebench"])
shown[[c for c in shown if c.startswith("accuracy")]] *= 100
display(shown.style.format("{:.0f}"))
md_ = shown.loc["muse_direct"]
display(Markdown(
    f"`muse_direct` names the right file for **{md_['accuracy, not shown']:.0f}%** of the "
    f"{md_['items, not shown']:.0f} items whose gold file wasn't in its context. "
    "Muse's knowledge of these repos (and the issue text) carries much of SWE-bench's accuracy. "
    "That is why Muse with only the memory summaries can match it, and why LoCoMo, "
    "where the answer exists only in the team's conversation, is the stricter test of memory."))

## 6. Routing headroom: the best any gate could do

A gate only chooses between answers that already exist: Llama's draft (`llama_only`), or an escalation. The **oracle** knows which ones are right and picks the cheapest right one: the draft if it's right, else the compact escalation, else (if it was run) the raw one. If none is right, it keeps the draft. No gate can beat it with these routes. A gate's accuracy can only reach `muse_direct`'s if the oracle's does.

In [ ]:
def oracle(bench: str, routes: list[str]) -> dict:
    correct, tokens = wide(bench).astype(bool), wide(bench, "muse_tokens")
    picked = pd.Series(routes[0], index=correct.index)
    for route in reversed(routes):  # the cheapest right route wins
        picked[correct[route]] = route
    acc = np.mean([correct.at[i, r] for i, r in picked.items()]) * 100
    spent = sum(tokens.at[i, r] for i, r in picked.items())
    return {"accuracy": acc, "token_share": spent / tokens.muse_direct.sum(),
            "escalated": (picked != routes[0]).mean() * 100}


rows = []
for bench in BENCHES:
    for c in conditions[bench]:
        r = star.loc[bench, c]
        rows.append({"bench": bench, "condition": c, "accuracy": r.accuracy,
                     "token_share": r.token_share, "escalated": cost.loc[(bench, c), "escalated"]})
    routes = ["llama_only", "escalate_compact"]
    rows.append({"bench": bench, "condition": "oracle: draft or compact", **oracle(bench, routes)})
    if "escalate_raw" in conditions[bench]:
        rows.append({"bench": bench, "condition": "oracle: draft, compact or raw",
                     **oracle(bench, routes + ["escalate_raw"])})
headroom = pd.DataFrame(rows).set_index(["bench", "condition"])
headroom.style.format({"accuracy": "{:.1f}", "token_share": "{:.1%}", "escalated": "{:.0f}%"})

In [ ]:
rows = []
for bench in BENCHES:
    correct = wide(bench).astype(bool)
    both = pd.crosstab(correct.llama_only.map({True: "draft right", False: "draft wrong"}),
                       correct.escalate_compact.map({True: "compact right", False: "compact wrong"}))
    display(Markdown(f"**{bench}**: Llama's draft vs the compact escalation, items"))
    display(both)
    lost = correct.muse_direct & ~correct[[c for c in correct if c != "muse_direct"]].any(axis=1)
    display(Markdown(f"{lost.sum()} {bench} items are right only with `muse_direct`: "
                     "no gateway route answers them."))

In [ ]:
gate_file = RESULTS / f"gate_{RUN_ID}.json"
gate = json.loads(gate_file.read_text()) if gate_file.exists() else None
markers = {"muse_direct": "s", "oracle: draft or compact": "D", "oracle: draft, compact or raw": "D"}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
for ax, bench in zip(axes, BENCHES):
    t = headroom.loc[bench]
    ref = t.accuracy["muse_direct"]
    ax.axhspan(ref - MARGIN, 100, color=GRID, alpha=0.5, lw=0)
    ax.axvline(BUDGET * 100, color=MUTED, lw=1, ls="--")
    if gate:
        sweep = pd.DataFrame([{"share": p[bench]["muse_token_share"] * 100,
                               "accuracy": p[bench]["accuracy"] * 100} for p in gate["sweep"]])
        sweep = sweep.sort_values("share")
        ax.plot(sweep.share, sweep.accuracy, color=AQUA, lw=2, label="learned gate, λ sweep", zorder=2)
    # Conditions at the same point (e.g. a gate that never escalates) share one label.
    for (share, acc), names in t.groupby([t.token_share.round(4), t.accuracy.round(2)]).groups.items():
        hollow = names[0].startswith("oracle")
        ax.scatter(share * 100, acc, s=70, marker=markers.get(names[0], "o"), zorder=3,
                   facecolor=SURFACE if hollow else BLUE, edgecolor=BLUE, linewidth=2)
        left = share > 0.5  # label the right-most points below on their left
        ax.annotate(", ".join(names), (share * 100, acc), xytext=(-8 if left else 7, -14 if left else -3),
                    textcoords="offset points", ha="right" if left else "left", fontsize=8, color=INK_2)
    top = max(t.accuracy.max(), ref) + 3
    ax.set_ylim(min(t.accuracy.min(), ref - MARGIN) - 3, top)
    ax.set_xscale("symlog", linthresh=1)
    ax.set_xlim(-0.3, 400)
    ax.set_xticks([0, 1, 10, 100], ["0%", "1%", "10%", "100%"])
    ax.set_xlabel("Muse tokens, % of muse_direct (log above 1%)")
    ax.set_ylabel("accuracy (%)")
    ax.set_title(f"{bench}: cost vs accuracy", loc="left", fontsize=11)
    if gate:
        ax.legend(frameon=False, fontsize=8, loc="lower right")
fig.text(0.01, -0.02, "Grey band: within 3 points of muse_direct. Dashed line: 10% of its tokens. "
         "The north star is the band's part left of the line. Hollow diamonds: the oracle.",
         fontsize=8, color=MUTED)
plt.tight_layout()
plt.show()

In [ ]:
# Each gate's choices against the oracle. Routes from cheapest to dearest; a gate is only
# judged on the routes it could take.
ROUTE_CONDITION = {"local_memory": "llama_only", "escalate_compact": "escalate_compact",
                   "escalate_raw": "escalate_raw"}
OUTCOMES = ["right: the cheapest right route", "right, but a cheaper route was also right",
            "wrong: a route it didn't take was right", "wrong: no route was right"]
for bench in BENCHES:
    correct = wide(bench).astype(bool)
    chosen = wide(bench, "route")
    table = {}
    for g in [c for c in ("rule_gate", "full") if c in conditions[bench]]:
        routes = ["local_memory", "escalate_compact"] + (["escalate_raw"] if g == "full" and
                                                         "escalate_raw" in conditions[bench] else [])
        outcome = []
        for item, route in chosen[g].items():
            right = [r for r in routes if correct.at[item, ROUTE_CONDITION[r]]]
            if route in right:
                outcome.append(OUTCOMES[0] if route == right[0] else OUTCOMES[1])
            else:
                outcome.append(OUTCOMES[2] if right else OUTCOMES[3])
        table[g] = pd.Series(outcome).value_counts()
    display(Markdown(f"**{bench}: each gate's decisions against the oracle, items**"))
    display(pd.DataFrame(table).reindex(OUTCOMES).fillna(0).astype(int))

## 7. Team scaling (replay)

SWE-bench tasks on one repo, done by 1–16 simulated engineers, two tasks each. Per-engineer cost should fall as the team grows only if the team really shares work:
- **Shared** (blue): one team, with shared memory and cache.
- **No sharing** (orange): every engineer on their own.
- **Solid** lines are related tasks (a team working in one area); **dashed** lines are random tasks.

Pages ingested and compute tokens don't depend on timing. Seconds do: ingest runs 16 calls at a time and the GPU starts cold, so they are noisy.

In [ ]:
csvs = sorted(RESULTS.glob("replay_*.csv"))
replay = pd.concat([pd.read_csv(p).assign(file=p.name) for p in csvs])
replay = replay[~replay.simulated.astype(bool) & ~replay.muse_is_standin.astype(bool)]
# The latest real-model replay per (baseline, order): file names end in the run's timestamp.
replay = replay[replay.file == replay.groupby(["baseline", "order"]).file.transform("max")]
metrics = [("ingest_pages_per_user", "memory pages ingested / engineer"),
           ("compute_tokens_per_user", "Llama compute tokens / engineer"),
           ("muse_tokens_per_user", "Muse tokens / engineer"),
           ("ingest_seconds_per_user", "ingest seconds / engineer")]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
for ax, (metric, title) in zip(axes, metrics):
    for (baseline, order), sub in replay.groupby(["baseline", "order"]):
        sub = sub.sort_values("team_size")
        ax.plot(sub.team_size, sub[metric], color=ORANGE if baseline == "no_sharing" else BLUE,
                lw=2, marker="o", ms=5, ls="--" if order == "random" else "-",
                label=f"{'no sharing' if baseline == 'no_sharing' else 'shared'}, {order}")
    ax.set_xscale("log", base=2)
    ax.set_xticks(sorted(replay.team_size.unique()))
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax.set_xlabel("team size")
    ax.set_title(title, loc="left", fontsize=10)
    ax.set_ylim(bottom=0)
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

ends = replay.pivot_table(index=["order", "baseline"], columns="team_size",
                          values="ingest_pages_per_user")
ends["change, 1 → 16"] = ends[16] / ends[1] - 1
display(Markdown("**Pages ingested per engineer**"))
display(ends.style.format("{:.1f}").format("{:+.0%}", subset=["change, 1 → 16"]))
display(Markdown(f"Files: {', '.join(sorted(replay.file.unique()))}"))

## 8. The learned gate

Written by `scripts/gate_run.sh` (`eval/learn_gate.py`).
- **The model:** one logistic model per route, on 11 features of the question, the retrieved memory and Llama's draft.
- **Scored out of fold:** whole teams are held out, so every number is on conversations and repos the gate didn't train on.
- **λ:** for each λ the gate picks the route with the best P(right) − λ × estimated Muse dollars. Sweeping λ draws the curve.
- **Operating point:** `learn_gate` picks one λ for both benchmarks, the most accurate whose *pooled* Muse tokens stay within 10%. SWE-bench's `muse_direct` spends far more tokens than LoCoMo's, so the pooled limit lets SWE-bench overspend and leaves LoCoMo under-escalated.
- **λ per benchmark:** the north star is judged per benchmark, so the table also shows the most accurate λ from the same sweep within each benchmark's own 10%. A team would set its own λ against its own budget, the same way. The grid of λ is coarse, so these points are approximate.

In [ ]:
if gate is None:
    display(Markdown(f"**Not run yet.** After `scripts/gate_run.sh`, `{gate_file.name}` appears; rerun "
                     "the notebook. The `full` condition then shows up in every table above."))
else:
    point = gate["operating_point"]
    sweep = pd.DataFrame([{"lam": p["lam"], "bench": b, "accuracy": p[b]["accuracy"] * 100,
                           "token_share": p[b]["muse_token_share"], "input_share": p[b]["muse_input_share"],
                           **{f"{r}": p[b]["routes"].get(r, 0.0) for r in gate["routes"]}}
                          for p in gate["sweep"] for b in BENCHES if b in p])
    ref = {b: star.loc[(b, "muse_direct"), "accuracy"] for b in BENCHES}

    def random_escalation(b, share):
        # Escalate a random fraction of questions down one route (in expectation), keep the
        # draft for the rest; the best such route at this token share. A gate must beat it.
        local = star.loc[(b, "llama_only")]
        best = local.accuracy
        for c in ("escalate_compact", "escalate_raw"):
            if c in conditions[b]:
                r = star.loc[(b, c)]
                best = max(best, local.accuracy + (r.accuracy - local.accuracy) * min(1, share / r.token_share))
        return best

    rows = []
    for b in BENCHES:
        s = sweep[sweep.bench == b]
        pooled = s[s.lam == point["lam"]].iloc[0]
        own = s[s.token_share <= BUDGET].sort_values(["accuracy", "lam"], ascending=False).iloc[0]
        for choice, r in (("one λ, pooled 10%", pooled), ("λ per benchmark, its own 10%", own)):
            rows.append({"bench": b, "choice": choice, "λ": r.lam, "accuracy": r.accuracy,
                         "gap": r.accuracy - ref[b], "random, same tokens": random_escalation(b, r.token_share),
                         "token_share": r.token_share,
                         "input_share": r.input_share, **{k: r[k] for k in gate["routes"]},
                         "verdict": "PASS" if r.accuracy - ref[b] >= -MARGIN and r.token_share <= BUDGET
                         else "FAIL"})
    choices = pd.DataFrame(rows).set_index(["bench", "choice"])
    display(Markdown(f"Routes {gate['routes']}, {gate['items']} items; "
                     f"{gate['retrieval_mismatches']} retrievals differ from the run's. "
                     "Route columns: share of questions sent down each route."))
    display(choices.style.format({"λ": "{:.3g}", "accuracy": "{:.1f}", "gap": "{:+.1f}",
                                  "random, same tokens": "{:.1f}",
                                  "token_share": "{:.1%}", "input_share": "{:.1%}",
                                  **{k: "{:.0%}" for k in gate["routes"]}}))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    for ax, b in zip(axes, BENCHES):
        s = sweep[sweep.bench == b].sort_values("token_share")
        ax.axhspan(ref[b] - MARGIN, 100, color=GRID, alpha=0.5, lw=0)
        ax.axvline(BUDGET * 100, color=MUTED, lw=1, ls="--")
        ax.plot(s.token_share * 100, s.accuracy, color=AQUA, lw=2, marker="o", ms=3,
                label="learned gate, λ sweep")
        h = headroom.loc[b]
        local = h.loc["llama_only"]
        for i, c in enumerate(c for c in ("escalate_compact", "escalate_raw") if c in h.index):
            ax.plot([0, h.loc[c].token_share * 100], [local.accuracy, h.loc[c].accuracy], color=MUTED,
                    lw=1, ls=":", label="random escalation" if i == 0 else None)
        best_oracle = [c for c in h.index if c.startswith("oracle")][-1]
        for c in ["llama_only", "rule_gate", "escalate_compact", "escalate_raw", best_oracle]:
            if c in h.index and not (c == "rule_gate" and h.loc[c].equals(h.loc["llama_only"])):
                r = h.loc[c]
                ax.scatter(r.token_share * 100, r.accuracy, s=60, zorder=3, marker="D" if c == best_oracle else "o",
                           facecolor=SURFACE if c == best_oracle else BLUE, edgecolor=BLUE, linewidth=2)
                ax.annotate(c, (r.token_share * 100, r.accuracy), xytext=(6, 4), textcoords="offset points",
                            fontsize=8, color=INK_2)
        o = choices.loc[(b, "λ per benchmark, its own 10%")]
        ax.scatter(o.token_share * 100, o.accuracy, s=90, zorder=4, color=AQUA, edgecolor=SURFACE, linewidth=2)
        ax.annotate("gate at its own 10%", (o.token_share * 100, o.accuracy), xytext=(6, -12),
                    textcoords="offset points", fontsize=8, color=INK_2)
        ax.set_xlim(0, 40)
        ax.set_ylim(min(s.accuracy.min(), ref[b] - MARGIN) - 4, max(ref[b], h.accuracy.max()) + 4)
        ax.set_xlabel("Muse tokens, % of muse_direct")
        ax.set_ylabel("accuracy (%)")
        ax.set_title(f"{b}: the learned gate vs the oracle (muse_direct {ref[b]:.1f}%)", loc="left", fontsize=11)
    axes[0].legend(frameon=False, fontsize=8, loc="lower right")
    fig.text(0.01, -0.02, "Grey band: within 3 points of muse_direct. Dashed line: 10% of its tokens. "
             "Hollow diamond: the oracle over every route. Dotted: escalating questions at random.", fontsize=8, color=MUTED)
    plt.tight_layout()
    plt.show()

## 9. Summary

Generated from the numbers above, so it stays current if anything is rerun.

In [ ]:
lines = []
for bench in BENCHES:
    t = star.loc[bench]
    ref = t.accuracy["muse_direct"]
    best = t.drop("muse_direct").sort_values(["gap", "token_share"], ascending=[False, True]).iloc[0]
    cheap = t[(t.index != "muse_direct") & (t.token_share <= BUDGET)]
    best_cheap = cheap.sort_values("gap", ascending=False).iloc[0] if len(cheap) else None
    oracle_name = [c for c in headroom.loc[bench].index if c.startswith("oracle")][-1]
    o = headroom.loc[(bench, oracle_name)]
    lines.append(
        f"- **{bench}**: `muse_direct` {ref:.1f}%. Most accurate gateway condition: "
        f"`{best.name}` {best.accuracy:.1f}% ({best.gap:+.1f} pts) at {best.token_share:.1%} of its tokens "
        f"({best.input_share:.1%} of its prompt tokens): {best.verdict}."
        + (f" Best within 10%: `{best_cheap.name}` {best_cheap.accuracy:.1f}% ({best_cheap.gap:+.1f} pts)."
           if best_cheap is not None else "")
        + f" The best oracle (`{oracle_name}`) reaches {o.accuracy:.1f}% at {o.token_share:.1%}.")
display(Markdown("\n".join(lines)))

### Reading these results (30 Oct, after the learned gate)

Section 9's summary is generated and stays current. This text was written from the numbers above on 30 Oct.

**The north star is not met on either benchmark.** Four findings explain why, and what holds up.

**1. On code, memory lets Muse match full context with under a tenth of the context.**
- **Results:** with the top-5 memory summaries, Muse scores 86.3% against `muse_direct`'s 86.0%, using 8.9% of its prompt tokens. With the raw pages behind them, it scores 87.0%, using 26%.
- **Why it still fails the 10% limit:** both fail on *total* tokens (23% and 34%), because Muse writes ~2,800 reasoning tokens per escalation however short the prompt. Passing on total tokens would need fewer escalations, which is the gate's job (finding 3).
- **Caveat:** `muse_direct` names the right file for 80% of questions whose gold file it never saw. Much of SWE-bench's accuracy is Muse's knowledge of these repos.

**2. On conversations, raw pages fix the summaries; retrieval is what's left.**
- **Raw vs summaries:** when retrieval finds the evidence's session, Muse with the raw pages scores **85.3%**, close to `muse_direct`'s 88.9%. With summaries it scores 60.5%. Overall, `escalate_raw` lifts LoCoMo from 46.0% to 62.7%, at 15.7% of `muse_direct`'s tokens.
- **The remaining loss is retrieval:** the top-5 memories miss the evidence's session for 37% of questions, and on those every gateway route scores 15–24%.
- **Hardest question types:** multi-hop (37% vs 74%), where the evidence is spread over sessions the top 5 rarely all cover.

**3. The learned gate, as built, is barely better than escalating at random.**
- **Like-for-like:** at the same Muse spend, compare the gate with sending a random share of questions to Muse (the dotted lines in section 8, and the "random, same tokens" column). On SWE-bench it is within a point of random everywhere. On LoCoMo it is 1–4 points better, mostly at low spend.

| Benchmark | Gate operating point | Accuracy | Random escalation, same tokens |
|---|---|---|---|
| SWE-bench | pooled | 79.3% at 18.2% | 80.0% |
| SWE-bench | its own 10% | 60.0% at 3.5% | 60.4% |
| LoCoMo | pooled | 46.3% at 2.5% | 42.2% |
| LoCoMo | its own 10% | 54.3% at 9.8% | 53.5% |

- **Not overfitting:** these numbers are out of fold, so the problem is signal. Features of the question, the retrieved memory and Llama's draft don't tell right drafts from wrong ones.
- **Compared with the rule gate:** on SWE-bench it sits on the same line. On LoCoMo the rule gate never escalates.

**4. The opportunity is real, and it is a signal problem.**
- **The oracle:** knowing which answers are right, it reaches **90.3% at 8.8%** of the tokens on SWE-bench, which passes the north star. On LoCoMo it reaches 66.7% at 3.6%.
- **Why:** on SWE-bench, Llama's draft is right for 167 of 300 questions, and escalating with summaries is wrong on only 6 of those.
- **What's missing:** a reliable signal that the draft is right. Cascade work (FrugalGPT) gets it from the cheap model's own confidence: the token log-probabilities of its answer, or a second sample that agrees with it. Both need Llama runs, but no Muse.
- **What we tried:** a feature from the answers already recorded is weak. Llama is right 62% of the time when its answer names a retrieved file, against 38% when it doesn't.

**Team scaling holds.**
- **Result:** with shared memory, pages ingested per engineer (the gateway's main GPU work) fall 35% at 16 engineers on related tasks, against 13% without sharing. On random tasks the drop is 7% vs 1%.
- **Unchecked:** the no-sharing, related, n = 16 Muse tokens (23K per engineer). Check them before quoting replay Muse tokens.

**Claims the results support:**
- (a) On code, shared memory lets the closed model match full-context accuracy with < 10% of the context.
- (b) On long conversations, raw retrieved pages recover the closed model's accuracy when retrieval finds the right session. Retrieval is the bottleneck.
- (c) Sharing memory across a team cuts per-engineer ingest work as the team grows.
- (d) A gate built on question, memory and draft features doesn't beat random escalation. The oracle shows ~30 points of headroom at the same cost on code, so the gate needs a draft-confidence signal.